<div style="background: linear-gradient(135deg, #005f73 0%, #0a9396 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">⚖️ 04 — O Dilema Viés-Variância</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 02 · Bloco 01 · Fundamentos</p>
</div>

## 🎯 Objetivos deste notebook

- Visualizar overfitting e underfitting com exemplos reais
- Entender o trade-off viés vs variância
- Diagnosticar problemas olhando curvas de aprendizado

---

## 1️⃣ O Problema Central do ML

| Situação | Nome | Sinal | Causa |
|---|---|---|---|
| Modelo simples demais | **Underfitting** | Loss alta em treino E teste | Viés alto |
| Modelo complexo demais | **Overfitting** | Loss baixa no treino, alta no teste | Variância alta |
| Equilíbrio | **Bom ajuste** | Loss baixa em ambos | ✅ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Gerar dados: y = sin(x) + ruído
np.random.seed(42)
X = np.sort(np.random.uniform(0, 6, 30)).reshape(-1, 1)
y = np.sin(X).ravel() + np.random.normal(0, 0.2, 30)
X_test = np.linspace(0, 6, 200).reshape(-1, 1)
y_test_real = np.sin(X_test).ravel()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titulos = ['Underfitting (grau 1)', 'Bom Ajuste (grau 4)', 'Overfitting (grau 15)']
graus = [1, 4, 15]

for ax, grau, titulo in zip(axes, graus, titulos):
    poly = PolynomialFeatures(grau)
    X_poly = poly.fit_transform(X)
    X_test_poly = poly.transform(X_test)
    model = LinearRegression().fit(X_poly, y)
    y_pred = model.predict(X_test_poly)
    mse_train = mean_squared_error(y, model.predict(X_poly))
    
    ax.scatter(X, y, alpha=0.6, s=40, label='Dados')
    ax.plot(X_test, y_pred, 'r-', linewidth=2, label=f'Modelo (grau {grau})')
    ax.plot(X_test, y_test_real, 'g--', alpha=0.5, label='Função real')
    ax.set_title(f'{titulo}\nMSE treino: {mse_train:.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(-2, 2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2️⃣ Curvas de Aprendizado: O Diagnóstico Definitivo

Plotar loss no **treino** e na **validação** conforme aumentamos os dados de treino:

In [ ]:
from sklearn.model_selection import learning_curve

poly = PolynomialFeatures(4)
X_poly = poly.fit_transform(X)
model = LinearRegression()

train_sizes, train_scores, val_scores = learning_curve(
    model, X_poly, y, cv=5, 
    train_sizes=np.linspace(0.2, 1.0, 8),
    scoring='neg_mean_squared_error'
)

train_mse = -train_scores.mean(axis=1)
val_mse = -val_scores.mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_mse, 'o-', color='#3498db', label='Treino', linewidth=2)
ax.plot(train_sizes, val_mse, 's-', color='#e74c3c', label='Validação', linewidth=2)
ax.fill_between(train_sizes, train_mse, val_mse, alpha=0.1, color='red')
ax.set_xlabel('Tamanho do Treino', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('Curva de Aprendizado — Gap = Overfitting', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.show()

print('→ Se o gap é grande: OVERFITTING (mais dados ou modelo mais simples)')
print('→ Se ambos são altos: UNDERFITTING (modelo mais complexo)')

## 🏋️ Questões para Praticar

### Q1. Repita com grau 20 e veja como a curva de aprendizado muda.

### Q2. Treine Decision Tree com `max_depth=None` vs `max_depth=3`. Qual sofre mais overfitting?

### Q3. Adicione ruído aos dados (mais noise). Como isso afeta o overfitting?

In [ ]:
# Resolva aqui
